# DistilBERT ATT&CK tactic training

Run this notebook with a Colab GPU runtime. It rebuilds the ATT&CK dataset, uses the repository's fixed source-grouped split, trains DistilBERT, and downloads a small evidence ZIP. The model checkpoint can optionally be copied to Google Drive.

In [ ]:
%cd /content
!test -d cyber/.git || git clone https://github.com/SahilBh01r1769/cyber.git
%cd /content/cyber
!git pull --ff-only

In [ ]:
!python -m pip install -q -e '.[transformer]'
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python -m src.data.build_dataset
!python -m src.data.split

In [ ]:
# Expected runtime depends on the assigned GPU. Do not interrupt while a training epoch is active.
!python -m src.models.train_transformer

In [ ]:
# Keep the large checkpoint in Drive; download only the small evidence archive for review.
SAVE_MODEL_TO_DRIVE = True

from pathlib import Path
import shutil
from google.colab import drive, files

evidence = Path('/content/cyber/artifacts/transformer_evidence.zip')
assert evidence.exists(), 'Training did not finish or the evidence archive was not created.'

if SAVE_MODEL_TO_DRIVE:
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/cti_attack_transformer')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(evidence, destination / evidence.name)
    shutil.copytree(
        '/content/cyber/artifacts/transformer_checkpoints/best_model',
        destination / 'best_model',
        dirs_exist_ok=True,
    )
    print('Saved to', destination)

files.download(str(evidence))